[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S13/2_Practica/reto_expres/Reto_expres_6_cancelaciones.ipynb)

# Reto exprés S12-13 · Clasificación: ¿se cancelará la reserva?

**Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · comparar tamaños de red **y la regresión logística**, y elegir **mirando solo la validación**
- ✏️ **2** · elegir el **umbral** según lo que cuesta no avisar a quien cancela y avisar a quien no
- ✏️ **3** · contadlo: titular con euros, matriz y curva de coste
- ✏️ **4** · la aplicación: probar una reserva
- Al final sale una **FICHA** con todos los números para vuestra presentación.

| Qué se predice | Datos | Rival simple | Métrica |
|---|---|---|---|
| si una reserva se cancela | 6 000 reservas (`reservas_turisdata.csv`) | regresión logística | AUC (más alto = mejor) y coste en € |

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, re, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEMILLA = 42

from tensorflow import keras
keras.utils.set_random_seed(SEMILLA)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
print("Listo ✔")

### ⚙️ Preparar las reservas — pulsa ▶

**Entrenamiento** (aprende), **validación** (para elegir) y **prueba** (una sola vez, 25 %). Las columnas de texto se convierten en columnas de 0 y 1.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"]); y = reservas["cancelada"].to_numpy()
X_res, X_pru, y_res, y_pru = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)
X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.25, stratify=y_res, random_state=SEMILLA)
cat = X.select_dtypes("object").columns.tolist(); num = [c for c in X.columns if c not in cat]
prep = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)]).fit(X_ent)
f32 = lambda D: np.asarray(prep.transform(D).todense() if hasattr(prep.transform(D), "todense") else prep.transform(D), dtype="float32")
A_ent, A_val, A_pru = f32(X_ent), f32(X_val), f32(X_pru)
print(f"{len(reservas)} reservas · {A_ent.shape[1]} columnas tras el preparado · se cancela el {y.mean():.1%}")
print(f"LÍNEA BASE: decir «no se cancela» a todas acierta el {(1 - y_pru.mean()):.1%} (pero no detecta ninguna cancelación).")

### ⚙️ Las candidatas — pulsa ▶

- **modelo simple**: regresión logística. Es el rival a batir.
- **red**: una capa oculta de N neuronas.

In [ ]:
def entrenar(neuronas, epocas=60):
    keras.utils.set_random_seed(SEMILLA)
    m = keras.Sequential([keras.layers.Input((A_ent.shape[1],)), keras.layers.Dense(neuronas, activation="relu"), keras.layers.Dropout(0.2), keras.layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    h = m.fit(A_ent, y_ent, validation_data=(A_val, y_val), epochs=epocas, batch_size=64, callbacks=[parar], verbose=0); return m, h
simple = LogisticRegression(max_iter=1000).fit(A_ent, y_ent)
print("Listo ✔")

### ✏️ 1 · Probar tamaños y elegir (solo con la validación)

Cambiad **PRUEBAS** (mínimo 3 valores). Mirad **val_auc** (0,5 = azar; más alto = mejor) y comparadlo con el **modelo simple**. Poned **ELEGIDA = 0** si os quedáis con la regresión logística.

In [ ]:
PRUEBAS = [4, 16, 64]      # neuronas de la capa oculta
filas = [dict(candidata="regresión logística (rival)", val_auc=round(float(roc_auc_score(y_val, simple.predict_proba(A_val)[:, 1])), 3), epocas="-", segundos="-")]
for n in PRUEBAS:
    t0 = time.time(); m, h = entrenar(n)
    filas.append(dict(candidata=f"red de {n} neuronas", val_auc=round(float(roc_auc_score(y_val, m.predict(A_val, verbose=0).ravel())), 3), epocas=len(h.history["loss"]), segundos=round(time.time() - t0)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 16               # ✏️ neuronas elegidas (o 0 para la regresión logística)

### ✏️ 2 · El umbral: ¿qué cuesta más? (solo con la validación)

El modelo da una **probabilidad** de cancelación. Se avisa (llamada, recordatorio, oferta) a quien supera el **umbral**.

- **COSTE_ESCAPA**: una cancelación que no se avisa (€ de una noche vacía).
- **COSTE_AVISO**: avisar a alguien que no iba a cancelar (€ de descuento o de molestia).

In [ ]:
COSTE_ESCAPA = 50        # ✏️ € por cancelación que se escapa
COSTE_AVISO = 30         # ✏️ € por aviso innecesario
if ELEGIDA > 0: modelo, hist = entrenar(ELEGIDA); predecir = lambda A: modelo.predict(A, verbose=0).ravel()
else: predecir = lambda A: simple.predict_proba(A)[:, 1]
p_val = predecir(A_val)
def coste(y, p, u):
    av = p >= u; return int((av & (y == 0)).sum()) * COSTE_AVISO + int((~av & (y == 1)).sum()) * COSTE_ESCAPA
umbrales = np.round(np.arange(0.1, 0.91, 0.05), 2)
costes = [coste(y_val, p_val, u) for u in umbrales]
display(pd.DataFrame(dict(umbral=umbrales, coste_eur=costes)).iloc[::2].T)
print("Sin modelo: no avisar a nadie =", coste(y_val, p_val, 2), "€ · avisar a todos =", coste(y_val, p_val, 0), "€")
UMBRAL = 0.4             # ✏️ umbral elegido (mirad dónde baja más el coste)

### ⚙️ La prueba, una sola vez — pulsa ▶

In [ ]:
p_pru = predecir(A_pru); auc_m = roc_auc_score(y_pru, p_pru); auc_s = roc_auc_score(y_pru, simple.predict_proba(A_pru)[:, 1])
av = p_pru >= UMBRAL; cm = confusion_matrix(y_pru, av, normalize="true")
c_modelo, c_nadie, c_todos = coste(y_pru, p_pru, UMBRAL), coste(y_pru, p_pru, 2), coste(y_pru, p_pru, 0)
FICHA = dict(elegida=ELEGIDA, umbral=UMBRAL, auc_simple=round(auc_s, 3), auc=round(auc_m, 3), recall=round(float(cm[1, 1]), 3), falsas_alarmas=round(float(cm[0, 1]), 3),
             coste=c_modelo, coste_nadie=c_nadie, coste_todos=c_todos, ahorro=c_nadie - c_modelo, reservas=len(y_pru))
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ predecir si la reserva se cancela · {len(reservas)} reservas
Candidatas ...... {[(f['candidata'], f['val_auc']) for f in filas]}  (AUC en validación)
Elegida ......... {"red de %d neuronas" % ELEGIDA if ELEGIDA > 0 else "regresión logística"} · umbral {UMBRAL}   (costes: escapa {COSTE_ESCAPA} €, aviso {COSTE_AVISO} €)
--- PRUEBA (una sola vez; {len(y_pru)} reservas) ---
AUC rival {auc_s:.3f} · AUC elegida {auc_m:.3f}   {"(la red gana)" if auc_m > auc_s + 0.01 else "(la red NO gana de forma clara)"}
Detecta el {cm[1, 1]:.0%} de las cancelaciones con un {cm[0, 1]:.0%} de falsas alarmas
Coste {c_modelo} € · sin modelo {c_nadie} € · avisando a todos {c_todos} €  → ahorro {c_nadie - c_modelo} €
============================================================
""")

### ✏️ 3 · Contadlo (S13): titular con euros, matriz y curva de coste

Cambiad **TITULAR**. Izquierda: **qué detecta y qué falsas alarmas da** (cada fila suma 100 %). Derecha: **cuánto cuesta cada umbral** en la prueba (eje desde 0).
**Cómo se cuenta (S13):** el titular es una **conclusión con cifra** (no «Resultados»), el gráfico empieza en **0** y las cifras salen de variables (f-string), no se teclean a mano. Terminad con un **límite**: ¿cuándo no os fiaríais?

In [ ]:
TITULAR = f"Avisando con umbral {UMBRAL} detectamos {cm[1, 1]:.0%} de las cancelaciones y ahorramos {c_nadie - c_modelo:,} € en {len(y_pru):,} reservas".replace(",", " ")   # ✏️ vuestra conclusión
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9)); ax[0].imshow(cm, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2): ax[0].text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", fontsize=15, color="white" if cm[i, j] > .5 else "black")
ax[0].set_xticks([0, 1], ["no avisa", "avisa"]); ax[0].set_yticks([0, 1], ["no cancela", "cancela"]); ax[0].set_xlabel("Lo que hace el modelo"); ax[0].set_ylabel("Lo que ocurrió")
cp = [coste(y_pru, p_pru, x) for x in umbrales]; ax[1].plot(umbrales, cp, marker="o"); ax[1].axvline(UMBRAL, color="black", ls="--", label=f"vuestro umbral ({UMBRAL})")
ax[1].axhline(c_nadie, color="tab:red", ls=":", label="sin modelo"); ax[1].set_ylim(0, None); ax[1].set_xlabel("umbral"); ax[1].set_ylabel("coste en la prueba (€)"); ax[1].legend()
fig.suptitle(TITULAR, fontsize=11, fontweight="bold"); plt.tight_layout(); plt.show()
FICHA["titular"] = TITULAR

### ✏️ 4 · La aplicación: probar una reserva

`probar(i, columna=valor, ...)` coge la reserva `i` de la prueba, cambia lo que le digáis y enseña la probabilidad de cancelación. Probad a subir el depósito o cambiar la tarifa.

In [ ]:
def probar(i=0, **cambios):
    fila = X_pru.iloc[[i]].copy()
    for k, v in cambios.items(): fila[k] = v
    p = float(predecir(f32(fila))[0])
    print(f"reserva {i} {cambios or ''} → probabilidad de cancelar {p:.0%} · " + ("AVISAR" if p >= UMBRAL else "no avisar"))

probar(0); probar(0, deposito="total", tarifa="no_reembolsable"); probar(0, antelacion_dias=5)

### 🎤 Para la presentación (2–3 min)

Copiad la **FICHA**, la tabla de la celda ✏️ 1 y el gráfico. Una frase para cada pregunta: **¿gana a la regresión logística? · ¿qué umbral y por qué? · ¿cuánto ahorra? · ¿a quién no deberíamos negarle una reserva por esto?**